# 🤗 Hugging Face Tokenizers

## **Tokenizers**
A **tokenizer** is a component in Natural Language Processing (NLP) that:

> **Converts raw text into numerical tokens** so it can be processed by a neural network.

In simple words:  
**Tokenizer = Text → Numbers**



##  **Token**

A **token** is:

> The smallest unit of text that a model processes.

Depending on the method, a token can be:
- A **word**
- A **character**
- A **subword**

Example:
```text
"learning"
→ ["learn", "##ing"]
```
## **Token ID**

A token ID is:

> A unique integer assigned to each token in the tokenizer’s vocabulary.

Example:
```text
"love" → 2293
```
Neural networks work only with these numbers, not raw text.

## **Vocabulary**

The vocabulary is:

> The complete list of all tokens known to a tokenizer, each mapped to a token ID.

- Larger vocabulary = more memory

- Smaller vocabulary = better generalization

Subword tokenization keeps the vocabulary compact and efficient.

### **Why Do We Even Need Tokenizers?**

Neural networks **do not understand text**.

They only understand **numbers (tensors)**.

So this sentence:

> **"I love machine learning"**

Must first become something like:

> **[1045, 2293, 3698, 4083]**

📌 **A tokenizer is the bridge between:**
- Human language  
- Neural network input  

Without tokenization, **NLP models cannot work at all**.





## Types of Tokenization


### **1. Word-Level Tokenization (Old Approach)**

Word-level tokenization splits text into full words.

Example:

```text
"I love NLP"
→ ["I", "love", "NLP"]
```

**❌ Limitations:**

- Cannot handle unseen words

- Vocabulary grows too large
---

### **2. Character-Level Tokenization**

Character-level tokenization splits text into individual characters.

```text
"Hello world"
→ ["H", "e", "l", "l", "o", " ", "w", "o", "r", "l", "d"]
```

**❌ Limitations:**

- Very long sequences

- Slower learning of meaning

---
### **3. Subword Tokenization — Definition (Most Important ⭐)**

Subword tokenization splits words into meaningful parts.

```text
"tokenizers"
→ ["token", "##izers"]
```

**✅ Advantages:**

- Handles unknown words

- Smaller vocabulary

- Efficient training

**📌 Used by:**

- BERT

- GPT

- LLaMA

- T5

---

### **4.Special Tokens**

Special tokens are **predefined tokens with special meaning.**

Common ones:

```text
[CLS] → Classification token

[SEP] → Sentence separator

[PAD] → Padding

[UNK] → Unknown token
```

📌 These help the model understand structure, not just content.

---

### **Attention Mask**

An attention mask is:

> A binary mask telling the model which tokens are real and which are padding.

**1 → real token**  
**0 → padding token**

This prevents the model from learning from fake data.

---

### **Token Type IDs**

Token type IDs are used to:

> Distinguish between two sentences in tasks like Question Answering.

Example:

Sentence A → 0

Sentence B → 1



## Two Libraries in Hugging Face Ecosystem

## **1. transformers**

transformers is a high-level Hugging Face library used to:

- **Load pretrained models**

- **Load matching tokenizers**

Fine-tune and run inference easily

👉 Most commonly used in real projects.

---

## **2. tokenizers**

tokenizers is a low-level Hugging Face library used to:

- **Build custom tokenizers**

- **Control tokenization rules**

Achieve very fast performance (Rust backend)

👉 Used when creating custom NLP pipelines.

## PART 1 — Using a Pretrained Tokenizer

Imagine you are fine-tuning BERT.

⚠️ **You must use the same tokenizer that BERT was trained on.**

### Why?

Because:

- Token IDs must match the model’s embeddings

- Different tokenizer = ❌ wrong inputs

**Step 1 — Install**

In [ ]:
pip install transformers

**Step 2 — Load Tokenizer**

### What happens internally?

- Downloads vocabulary

- Loads normalization rules

- Loads special tokens

- Loads WordPiece model

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

**Step 3 — Tokenize Text**

- Lowercased automatically

- Punctuation handled

In [ ]:
text = "Hugging Face makes AI accessible."

tokens = tokenizer.tokenize(text)
print(tokens)

['hugging', 'face', 'makes', 'ai', 'accessible', '.']


**Step 4 — Convert to Model Input**

`input_ids` → numerical tokens

`attention_mask` → tells model which tokens are real vs padding

In [ ]:
encoded = tokenizer(text)

print("Input IDs:", encoded["input_ids"])
print("Attention Mask:", encoded["attention_mask"])

Input IDs: [101, 17662, 2227, 3084, 9932, 7801, 1012, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1]


**Step 5 — Special Tokens**

In [ ]:
print(tokenizer.cls_token)
print(tokenizer.sep_token)

[CLS]
[SEP]


## **PART 2 — Building a Tokenizer From Scratch**

Now assume:

You are training your own model for:

- Legal documents

- Medical notes

- Programming code

- Hindi language

Then you must build your own tokenizer.

**Step 1 — Install Tokenizers Library**

In [ ]:
pip install tokenizers

**Step 2 — Import Required Modules**

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase

## 🔁 Understanding the Tokenizer Pipeline

Tokenizer Pipeline:

Raw Text  
&nbsp;&nbsp;&nbsp;↓  
Normalizer  
&nbsp;&nbsp;&nbsp;↓  
PreTokenizer  
&nbsp;&nbsp;&nbsp;↓  
Model (BPE / WordPiece)  
&nbsp;&nbsp;&nbsp;↓  
PostProcessor  
&nbsp;&nbsp;&nbsp;↓  
IDs  

**Step 3- Normalizer**
Purpose:

- Lowercase

- Remove accents

- Unicode cleanup

In [ ]:
from tokenizers.normalizers import Sequence, Lowercase, StripAccents, NFD

tokenizer.normalizer = Sequence([
    NFD(),
    Lowercase(),
    StripAccents()
])

In [ ]:
print(tokenizer.normalizer.normalize_str("HéLLo"))

hello


### **What’s Happening Internally**

```text
HéLLo
 ↓
NFD()
HéLLo   (e + combining accent)
 ↓
Lowercase()
héllo
 ↓
StripAccents()
hello
```


**Step 4 — Add Pre-Tokenizer**

Splits text before model sees it.

In [ ]:
tokenizer.pre_tokenizer = Whitespace()

In [ ]:
text = "Hello world! How are you?"
output = tokenizer.pre_tokenizer.pre_tokenize_str(text)

print(output)

[('Hello', (0, 5)), ('world', (6, 11)), ('!', (11, 12)), ('How', (13, 16)), ('are', (17, 20)), ('you', (21, 24)), ('?', (24, 25))]


**Step 5 — Create Tokenizer Model**

We use **BPE (Byte Pair Encoding)**.

Why BPE?

- Used in GPT models

- Good balance between word + character

Options:

- BPE → GPT-style

- WordPiece → BERT-style

- Unigram → SentencePiece-style

In this component:

- `vocab_size` controls vocabulary limit

- Special tokens are manually added

- BPE learns frequent patterns

In [ ]:
from tokenizers.trainers import BpeTrainer

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

trainer = BpeTrainer(
    vocab_size=100,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]"]
)

corpus = ["Hello World", "Hello there"]
tokenizer.train_from_iterator(corpus, trainer)

**Step 6 — PostProcessor**

Adds special tokens automatically.

For example in BERT:
```text
[CLS] sentence [SEP]

In [ ]:
from tokenizers.processors import TemplateProcessing

tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B [SEP]",
    special_tokens=[
        ("[CLS]", tokenizer.token_to_id("[CLS]")),
        ("[SEP]", tokenizer.token_to_id("[SEP]")),
    ],
)

**Step 7 — Test Encoding**

In [ ]:
output = tokenizer.encode("Hello", "World")

print("Tokens:", output.tokens)
print("IDs:", output.ids)

Tokens: ['[CLS]', 'Hell', 'o', '[SEP]', 'Wo', 'rld', '[SEP]']
IDs: [2, 17, 11, 3, 19, 23, 3]


**Step 8 — Decode Back**


Converts IDs → readable text.

- Removes special tokens

- Merges subwords properly

In [ ]:
decoded = tokenizer.decode(output.ids)
print(decoded)

Hell o Wo rld


**Step 9 — Save**

In [ ]:
tokenizer.save("customer_support_tokenizer.json")

In [ ]:
tokenizer = Tokenizer.from_file("customer_support_tokenizer.json")

## PART 3 — What Happens Inside BPE?

Let’s understand **Byte Pair Encoding (BPE)** step by step.

### Suppose the training corpus contains:

```text
low
lowest
lower
```

---

### 🔹 Step 1: Start with characters

Initially, every word is split into **characters**:

```text
l o w
l o w e s t
l o w e r
```

---

### 🔹 Step 2: Count frequent adjacent pairs

The pair **(l, o)** appears very frequently  
The pair **(lo, w)** also appears very frequently

---

### 🔹 Step 3: Merge the most frequent pairs

```text
l + o → lo
lo + w → low
```

Now the words look like:

```text
low
low e s t
low e r
```

---

### 🔹 Step 4: Continue merging

Frequent suffixes are learned:


`e + s + t → est`


So finally:

```text
lowest → low + est
lower → low + er
```

---

### ✅ Key Intuition

- Common roots are reused (`low`)
- Suffixes are learned separately (`est`, `er`)
- New words can be formed from known pieces

👉 **This is why subword tokenization works so well.**




## PART 4 — Comparing BPE vs WordPiece

| Feature | BPE | WordPiece |
|------|----|-----------|
| Used in | **:contentReference[oaicite:0]{index=0}** | **:contentReference[oaicite:1]{index=1}** |
| Merge rule | Frequency-based | Probability-based |
| Objective | Compress text efficiently | Maximize likelihood |
| Training speed | Fast | Moderate |
| Token prefix | None | `##` used for continuation |

---

## WordPiece Example

WordPiece **marks subword continuations** using `##`.

Example:


"playing"
→ ["play", "##ing"]


Another example:


"unbelievable"
→ ["un", "##believ", "##able"]


📌 `##` means:
> “This token is a continuation of the previous token”


In [ ]:
from tokenizers.models import WordPiece

tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

### **Code Simulation of BPE Merge**

In [ ]:
from collections import Counter

corpus = ["low", "lowest", "lower"]

# Convert to character representation
def word_to_chars(word):
    return list(word) + ["</w>"]

vocab = [word_to_chars(word) for word in corpus]

def get_pairs(vocab):
    pairs = Counter()
    for word in vocab:
        for i in range(len(word)-1):
            pairs[(word[i], word[i+1])] += 1
    return pairs

pairs = get_pairs(vocab)
print("Pair Frequencies:")
print(pairs)

Pair Frequencies:
Counter({('l', 'o'): 3, ('o', 'w'): 3, ('w', 'e'): 2, ('w', '</w>'): 1, ('e', 's'): 1, ('s', 't'): 1, ('t', '</w>'): 1, ('e', 'r'): 1, ('r', '</w>'): 1})


**Merge the most frequent pair manually(Optional)**

In [ ]:
def merge_pair(pair, vocab):
    new_vocab = []
    bigram = "".join(pair)
    for word in vocab:
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word)-1 and (word[i], word[i+1]) == pair:
                new_word.append(bigram)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_vocab.append(new_word)
    return new_vocab

In [ ]:
best = max(pairs, key=pairs.get)
vocab = merge_pair(best, vocab)
print(vocab)

[['lo', 'w', '</w>'], ['lo', 'w', 'e', 's', 't', '</w>'], ['lo', 'w', 'e', 'r', '</w>']]


## **Realistic Example**

In [ ]:
code_data = [
    "def train_model():",
    "for epoch in range(10):",
    "model.fit(X_train, y_train)"
]

**Step 1 — Build the Tokenizer Pipeline**

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Sequence, Whitespace, Punctuation
from tokenizers.normalizers import Lowercase

# 1️⃣ Create model
tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

# 2️⃣ Add stages
tokenizer.normalizer = Lowercase()

tokenizer.pre_tokenizer = Sequence([
    Whitespace(),
    Punctuation()
])

# 3️⃣ Trainer
trainer = WordPieceTrainer(
    vocab_size=50,
    special_tokens=["[UNK]", "[PAD]"]
)

# 4️⃣ Train
tokenizer.train_from_iterator(code_data, trainer)

**Step 2 — Visualize Each Stage**

**1. Raw Text**

In [ ]:
text = "model.fit(X_train, y_train)"

**2. After Normalization**

In [ ]:
print(tokenizer.normalizer.normalize_str(text))

model.fit(x_train, y_train)


**3. After Pre-Tokenization**

In [ ]:
print(tokenizer.pre_tokenizer.pre_tokenize_str(text))

[('model', (0, 5)), ('.', (5, 6)), ('fit', (6, 9)), ('(', (9, 10)), ('X', (10, 11)), ('_', (11, 12)), ('train', (12, 17)), (',', (17, 18)), ('y', (19, 20)), ('_', (20, 21)), ('train', (21, 26)), (')', (26, 27))]


**4. After WordPiece**

In [ ]:
output = tokenizer.encode(text)

print("Tokens:", output.tokens)
print("IDs:", output.ids)

Tokens: ['model', '.', 'f', '##i', '##t', '(', 'x', '_', 'train', ',', 'y', '_', 'train', ')']
IDs: [49, 5, 14, 29, 30, 2, 25, 9, 45, 4, 26, 9, 45, 3]
